In [2]:
# %%
import extract_msg
import os
from bs4 import BeautifulSoup
import csv # Embora importado, csv não está sendo usado diretamente para escrita, pandas cuida disso.
import pandas as pd
from openai import OpenAI # A parte da OpenAI será comentada e precisará de configuração de API Key segura.
import re
import tiktoken

In [8]:
# %%
# Caminhos e nomes de arquivos
path_to_msg_folder = 'TeamsMessages\\TeamsMessages\\helpdesk@pecege.com (Primary)\\TeamsMessagesData' # Certifique-se que esta pasta existe e contém seus arquivos .msg
output_initial_report_csv = 'teams_messages_report_detailed.csv'
output_formatted_grouped_csv = 'teams_messages_grouped_conversations.csv'
# output_formatted_with_summaries_csv = 'teams_messages_with_summaries.csv' # Para uso futuro

# Lista para armazenar os dados extraídos
print("DEBUG: Inicializando/Re-inicializando messages_data...")
messages_data = []
print(f"DEBUG: messages_data agora tem {len(messages_data)} itens.")

DEBUG: Inicializando/Re-inicializando messages_data...
DEBUG: messages_data agora tem 0 itens.


In [9]:
# %%
# Definição ÚNICA do e-mail normalizado do Help Desk (usado em create_normalized_chat_id)
help_desk_normalized_email = 'helpdesk@pecege.com'
print(f"DEBUG: help_desk_normalized_email definido como: '{help_desk_normalized_email}'")


# Função para extrair e limpar o endereço de e-mail (VERSÃO APRIMORADA)
def extract_and_clean_email(text_field):
    if pd.isna(text_field):
        return None

    text_field_str = str(text_field).replace('\x00', '').strip()
    
    parts = text_field_str.split(';')
    valid_emails_found = []
    
    for part in parts:
        part = part.strip()
        if not part:
            continue

        # Ignora partes que são claramente placeholders como <None> ou não contêm email
        # mas não descarta a string inteira só por causa disso.
        if ('<none' in part.lower() or 'none>' in part.lower()) and '@' not in part:
            continue

        email_candidate_from_part = None
        match_angle = re.search(r'<([^>]+)>', part)

        if match_angle:
            candidate_in_angle = match_angle.group(1).strip()
            # Verifica se o conteúdo dentro dos colchetes parece um email e não é 'none'
            if '@' in candidate_in_angle and 'none' not in candidate_in_angle.lower():
                email_candidate_from_part = candidate_in_angle
        else:
            # Se não houver <>, a parte em si pode ser um email ou conter um.
            # Procura por palavras que se assemelhem a um e-mail.
            # Remove pontuações comuns que podem estar anexadas ao e-mail.
            sub_parts = part.split()
            possible_emails_in_sub_part = [
                sp.strip(',.;:"\'()[]{}') for sp in sub_parts if '@' in sp and '.' in sp
            ]
            if possible_emails_in_sub_part:
                # Pega o último candidato, pois nomes podem vir antes.
                email_candidate_from_part = possible_emails_in_sub_part[-1]

        if email_candidate_from_part:
            cleaned_candidate = email_candidate_from_part.strip().lower()
            # Validação estrita do formato do e-mail
            if re.fullmatch(r'([a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,})', cleaned_candidate):
                valid_emails_found.append(cleaned_candidate)

    if not valid_emails_found:
        return None

    # Lógica de priorização:
    # Se o e-mail do Help Desk estiver na lista de e-mails válidos:
    if help_desk_normalized_email in valid_emails_found:
        # Procure por outros e-mails (que seriam de clientes)
        other_emails = [e for e in valid_emails_found if e != help_desk_normalized_email]
        if other_emails:
            return other_emails[0] # Retorna o primeiro e-mail de cliente encontrado
        else:
            # Se apenas o e-mail do Help Desk foi encontrado (ex: cliente enviou para o HD)
            return help_desk_normalized_email
    # Se o e-mail do Help Desk NÃO estiver na lista, mas outros e-mails válidos sim:
    elif valid_emails_found:
        return valid_emails_found[0] # Retorna o primeiro e-mail de cliente encontrado
        
    return None





DEBUG: help_desk_normalized_email definido como: 'helpdesk@pecege.com'


In [10]:
# %%
# Processar os arquivos .msg
print(f"\nProcessando arquivos da pasta: {path_to_msg_folder}")
if not os.path.exists(path_to_msg_folder) or (os.path.exists(path_to_msg_folder) and not os.listdir(path_to_msg_folder)):
    print(f"AVISO: Pasta {path_to_msg_folder} não encontrada ou vazia.")
else:
    for file_name in os.listdir(path_to_msg_folder):
        if file_name.endswith(".msg"):
            file_path = os.path.join(path_to_msg_folder, file_name)
            # print(f"DEBUG: Processando arquivo: {file_path} ") # Descomente para log detalhado
            try:
                msg = extract_msg.Message(file_path)
                # print(f"  Sender: {msg.sender}") # Descomente para log detalhado
                # print(f"  To: {msg.to}\n")       # Descomente para log detalhado
                msg_sender_original = msg.sender
                msg_to_original = msg.to
                msg_date = msg.date
                msg_message_html = msg.htmlBodyPrepared

                if msg_message_html:
                    formatted_message_body = BeautifulSoup(msg_message_html, 'html.parser').get_text(separator=' ', strip=True)
                else:
                    formatted_message_body = msg.body
                    if formatted_message_body:
                        formatted_message_body = formatted_message_body.strip()
                    else:
                        formatted_message_body = ""
                
                messages_data.append({
                    'Original_From': msg_sender_original,
                    'Original_To': msg_to_original,
                    'Date': msg_date,
                    'Message_Body': formatted_message_body,
                    'Source_File': file_name
                })
            except Exception as e:
                print(f"Erro ao processar o arquivo {file_path}: {e}")
                continue


Processando arquivos da pasta: TeamsMessages\TeamsMessages\helpdesk@pecege.com (Primary)\TeamsMessagesData


In [11]:
from typing import Counter

print(f"\nDEBUG: Total de itens em messages_data após loop: {len(messages_data)}")
if messages_data:
    source_files_in_data = [item['Source_File'] for item in messages_data]
    file_counts = Counter(source_files_in_data)
    files_with_duplicates = {file: count for file, count in file_counts.items() if count > 1}
    if files_with_duplicates:
        print("DEBUG: Contagem de Source_File em messages_data (apenas os com duplicatas):")
        for file, count in files_with_duplicates.items():
            print(f"  ALERTA DE DUPLICIDADE: Arquivo '{file}' aparece {count} vezes em messages_data.")
    else:
        print("DEBUG: Nenhum Source_File duplicado encontrado em messages_data.")
else:
    print("DEBUG: messages_data está vazia após o loop.")

# %%
print(f"\nTotal de mensagens extraídas (final de messages_data): {len(messages_data)}")


DEBUG: Total de itens em messages_data após loop: 4389
DEBUG: Nenhum Source_File duplicado encontrado em messages_data.

Total de mensagens extraídas (final de messages_data): 4389


In [12]:
if not messages_data:
    print("Nenhuma mensagem foi extraída. Verifique a pasta de origem e os arquivos .msg.")
else:
    print(f"\nDEBUG: Criando DataFrame a partir de {len(messages_data)} itens em messages_data.")
    df = pd.DataFrame(messages_data)
    
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce', utc=True) 
    df.dropna(subset=['Date'], inplace=True)
    
    df_sorted = df.sort_values(by='Date').reset_index(drop=True)
    print(f"DEBUG: df_sorted criado com {len(df_sorted)} linhas.")

    # Limpeza de caracteres nulos ANTES de aplicar extract_and_clean_email
    colunas_de_texto_para_limpar = ['Original_From', 'Original_To', 'Message_Body'] 
    for coluna in colunas_de_texto_para_limpar:
        if coluna in df_sorted.columns:
            df_sorted[coluna] = df_sorted[coluna].astype(str).str.replace('\x00', '', regex=False)
    
    print("DEBUG: Aplicando extract_and_clean_email a 'Original_From' e 'Original_To'...")
    df_sorted['From_Email'] = df_sorted['Original_From'].apply(extract_and_clean_email)
    df_sorted['To_Email'] = df_sorted['Original_To'].apply(extract_and_clean_email)

    df_sorted.to_csv(output_initial_report_csv, index=False, encoding='utf-8-sig')
    print(f"Relatório detalhado inicial salvo em: {output_initial_report_csv}")

    # Criando um ChatID com base nos e-mails normalizados (VERSÃO DO USUÁRIO)
    def create_normalized_chat_id(row):
        from_email = row.get('From_Email')
        to_email = row.get('To_Email')
        row_name = row.name # índice da linha

        # Correção adicional se e-mails extraídos resultarem na string "none"
        if isinstance(to_email, str) and 'none' in to_email.lower(): # Verifica se é string antes de .lower()
            to_email = None
        if isinstance(from_email, str) and 'none' in from_email.lower():
            from_email = None
        
        # Lógica principal para ChatID
        if from_email == help_desk_normalized_email and pd.notna(to_email):
            # Mensagem DO Help Desk PARA um cliente (to_email)
            return to_email
        elif pd.notna(from_email) and from_email != help_desk_normalized_email:
            # Mensagem DE um cliente (from_email) PARA qualquer um (Help Desk ou outro)
            return from_email
        
        # Fallbacks para casos onde a lógica acima não se aplica (e-mails são NaN ou HD para NaN)
        # Se From é HD e To é NaN (não deveria acontecer se HD sempre responde a alguém)
        if from_email == help_desk_normalized_email and pd.isna(to_email):
             print(f"  DEBUG ChatID: HD para NaN. Linha {row_name}. Original_To: {repr(row.get('Original_To'))}")
             return f"unknown_hd_to_nan_{row_name}"
        
        # Outros casos de e-mails ausentes
        if pd.isna(from_email) and pd.isna(to_email):
            return f"unknown_no_emails_{row_name}"
        if pd.isna(from_email): # to_email existe mas não é HD (senão cairia no 1º if)
            return f"unknown_no_from_to_{str(to_email)}_{row_name}"
        if pd.isna(to_email): # from_email existe mas não é HD (senão cairia no 2º if ou no de HD para NaN)
             print(f"  DEBUG ChatID: Cliente para NaN. Linha {row_name}. From: {from_email}, Original_To: {repr(row.get('Original_To'))}")
             return f"unknown_client_to_nan_from_{str(from_email)}_{row_name}"

        # Fallback muito genérico, idealmente não deveria ser atingido com a lógica acima
        print(f"  DEBUG ChatID: Fallback genérico. Linha {row_name}. From: {from_email}, To: {to_email}")
        return f"generic_chat_{str(from_email)}_to_{str(to_email)}_{row_name}"

    print("DEBUG: Aplicando create_normalized_chat_id...")
    df_sorted['ChatID'] = df_sorted.apply(create_normalized_chat_id, axis=1)
    
    def format_display_message(row):
        original_from_cleaned = str(row['Original_From']) if pd.notna(row['Original_From']) else 'Remetente Desconhecido'
        message_body_cleaned = str(row['Message_Body']) if pd.notna(row['Message_Body']) else ''
        return f"[{row['Date'].strftime('%Y-%m-%d %H:%M:%S %Z')}] {original_from_cleaned}: {message_body_cleaned}"

    df_sorted['FormattedMessage'] = df_sorted.apply(format_display_message, axis=1)

    print("DEBUG: Filtrando para df_valid_client_chats...")
    # Filtra para ChatIDs que são emails válidos e não são o email do próprio Help Desk
    df_valid_client_chats = df_sorted[
        df_sorted['ChatID'].str.contains('@', na=False) & 
        (df_sorted['ChatID'] != help_desk_normalized_email)
        # IDs 'unknown...' que não contêm '@' serão naturalmente filtrados pela primeira condição.
        # IDs 'unknown...' que contêm '@' (ex: unknown_client_to_nan_from_cliente@ex.com_123)
        # passarão aqui, o que pode ser desejado ou não dependendo do objetivo de agrupamento.
    ]
    print(f"DEBUG: df_valid_client_chats tem {len(df_valid_client_chats)} linhas.")
    
    if df_valid_client_chats.empty:
        print("Nenhum chat de cliente válido encontrado para agrupamento. Verifique os dados e a lógica do ChatID.")
        grouped_conversations = pd.DataFrame(columns=['ChatID', 'ConversationHistory']) 
    else:
        print("DEBUG: Agrupando conversas...")
        grouped_conversations = df_valid_client_chats.groupby('ChatID')['FormattedMessage'].apply(lambda x: '\n'.join(x)).reset_index()
        grouped_conversations.rename(columns={'FormattedMessage': 'ConversationHistory'}, inplace=True)
        
        grouped_conversations.to_csv(output_formatted_grouped_csv, index=False, encoding='utf-8-sig')
        print(f"Conversas formatadas e agrupadas salvas em: {output_formatted_grouped_csv}")
        print(f"Total de conversas agrupadas (ChatIDs únicos de clientes): {grouped_conversations['ChatID'].nunique()}")



DEBUG: Criando DataFrame a partir de 4389 itens em messages_data.
DEBUG: df_sorted criado com 4389 linhas.
DEBUG: Aplicando extract_and_clean_email a 'Original_From' e 'Original_To'...
Relatório detalhado inicial salvo em: teams_messages_report_detailed.csv
DEBUG: Aplicando create_normalized_chat_id...
  DEBUG ChatID: HD para NaN. Linha 783. Original_To: '28:61b05713-15d4-4083-976e-e53b971f8aaf <None>'
  DEBUG ChatID: HD para NaN. Linha 785. Original_To: '28:61b05713-15d4-4083-976e-e53b971f8aaf <None>'
  DEBUG ChatID: HD para NaN. Linha 787. Original_To: '28:61b05713-15d4-4083-976e-e53b971f8aaf <None>'
  DEBUG ChatID: HD para NaN. Linha 789. Original_To: '28:61b05713-15d4-4083-976e-e53b971f8aaf <None>'
  DEBUG ChatID: HD para NaN. Linha 791. Original_To: '28:61b05713-15d4-4083-976e-e53b971f8aaf <None>'
  DEBUG ChatID: HD para NaN. Linha 793. Original_To: '28:61b05713-15d4-4083-976e-e53b971f8aaf <None>'
  DEBUG ChatID: HD para NaN. Linha 795. Original_To: '28:61b05713-15d4-4083-976e-e5

In [13]:
if 'grouped_conversations' in locals() and not grouped_conversations.empty:
    print("\nExibindo 'grouped_conversations' (primeiras linhas):")
    print(grouped_conversations.head())
elif 'grouped_conversations' in locals():
    print("\n'grouped_conversations' foi criado, mas está vazio.")
else:
    print("\n'grouped_conversations' não foi definido.")

# %%
if 'df_sorted' in locals() and not df_sorted.empty:
    print("\nValores únicos em 'From_Email' (após limpeza):")
    print(df_sorted['From_Email'].unique())
    print("\nValores únicos em 'To_Email' (após limpeza):")
    print(df_sorted['To_Email'].unique())
    print("\nValores únicos em 'ChatID' (gerados):")
    all_chat_ids = df_sorted['ChatID'].unique()
    unknown_chat_ids = [cid for cid in all_chat_ids if isinstance(cid, str) and cid.startswith('unknown')]
    client_chat_ids = [cid for cid in all_chat_ids if isinstance(cid, str) and '@' in cid and cid != help_desk_normalized_email and not cid.startswith('unknown')]
    other_chat_ids = [cid for cid in all_chat_ids if cid not in unknown_chat_ids and cid not in client_chat_ids]

    print(f"  ChatIDs 'unknown...': ({len(unknown_chat_ids)}): {unknown_chat_ids[:10]}...") # Mostra até 10
    print(f"  ChatIDs de Clientes (esperados): ({len(client_chat_ids)}): {client_chat_ids[:10]}...")
    if other_chat_ids:
        print(f"  Outros ChatIDs: ({len(other_chat_ids)}): {other_chat_ids[:10]}...")
else:
    print("\n'df_sorted' não foi definido ou está vazio.")


Exibindo 'grouped_conversations' (primeiras linhas):
                           ChatID  \
0      albertoraymundo@pecege.com   
1      alessandrapapik@pecege.com   
2  alexalmeida@pecegeprojetos.com   
3        alexdealmeida@pecege.com   
4      alfred@teams.microsoft.com   

                                 ConversationHistory  
0  [2025-03-11 16:49:06 UTC] Alberto Raymundo <al...  
1  [2025-02-25 19:03:50 UTC] Alessandra Natalia T...  
2  [2025-03-18 16:44:53 UTC] Alex Nunes de Almeid...  
3  [2025-03-10 13:32:09 UTC] Alex de Almeida <ale...  
4  [2025-02-27 12:11:57 UTC] alfred <alfred@teams...  

Valores únicos em 'From_Email' (após limpeza):
['helpdesk@pecege.com' 'marielleburin@pecege.com'
 'julianarozao@pecege.com' 'beatrizfranco@pecege.com'
 'laurapavani@pecege.com' 'jessicarodrigues@pecege.com'
 'hellenmeireles@pecege.com' 'marianogueira@pecege.com'
 'cesarstenico@pecege.com' 'guilhermebassa@pecege.com'
 'lorenamaniero@pecege.com' 'ricardotanaka@pecege.com'
 'matheusricci@pece

In [8]:
df_sorted[df_sorted['ChatID'].str.contains('unknown_chat_no_to_email')]


,Original_From,Original_To,Date,Message_Body,Source_File,From_Email,To_Email,ChatID,FormattedMessage


In [9]:
print("Valores únicos em 'From' antes da limpeza:")
print(df_sorted['From_Email'].unique())

Valores únicos em 'From' antes da limpeza:
['helpdesk@pecege.com' 'marielleburin@pecege.com'
 'julianarozao@pecege.com' 'beatrizfranco@pecege.com'
 'laurapavani@pecege.com' 'jessicarodrigues@pecege.com'
 'hellenmeireles@pecege.com' 'marianogueira@pecege.com'
 'cesarstenico@pecege.com' 'guilhermebassa@pecege.com'
 'lorenamaniero@pecege.com' 'ricardotanaka@pecege.com'
 'matheusricci@pecege.com' 'vivianebarros@pecege.com'
 'weslleisouza@pecege.com' 'beatrizsantos@pecege.com'
 'angelobatista@pecege.com' 'mariabraghetta@pecege.com'
 'felipejaoude@pecege.com' 'marcelamoreira@pecege.com'
 'silvanacavicchioli@pecege.com' 'arianesilva@pecege.com'
 'leonardobraga@pecege.com' 'alessandrapapik@pecege.com'
 'julianaoliveira@pecege.com' 'mariacecilia@pecege.com'
 'renata@pecege.com' 'jozecorrea@pecege.com' 'luanabonifacio@pecege.com'
 'leticiabarboza@pecege.com' 'gabrieleziviani@pecege.com'
 'feliperibeiro@pecege.com' 'liviafassiroli@pecege.com'
 'laillacarvalho@pecege.com' 'isabeladonderi@pecege.co

In [10]:
prompt_instructions = """Você é um analista inteligente de conversas de atendimento técnico.

Receberá abaixo uma transcrição de conversa entre um colaborador e o time de Help Desk. Sua tarefa é analisar e identificar **atendimentos reais de suporte**.

---

### Instruções:

1. **Identifique e separe atendimentos distintos**, baseando-se em:
   - Mudança de data;
   - Intervalos longos entre mensagens (mais de 60 minutos);
   - Novas mensagens que iniciam outro problema (ex: “bom dia, estou com outro problema…”).

2. **Ignore conversas que não representem uma solicitação de suporte técnico**, como:
   - Agradecimentos;
   - Confirmações de que o problema já foi resolvido;
   - Mensagens informativas sem solicitação de ação;
   - Mensagens genéricas ou elogios sem pedido de ajuda.

3. Considere como atendimento **válido** apenas quando a primeira mensagem da conversa expressar:
   - Um pedido de ajuda;
   - Um relato de problema;
   - Um sintoma ou erro observado (mesmo que sem pedir ajuda diretamente).

4. Para **cada atendimento válido identificado**, retorne os seguintes dados no formato JSON:

- `quem_solicitou_atendimento`: Nome completo da pessoa que solicitou suporte.
- `email_solicitante`: E-mail da pessoa que solicitou suporte.
- `quem_respondeu`: Nome(s) dos atendentes que se identificaram com algo como “Mateus aqui”, “Barbi aqui” etc.
- `data_solicitacao`: Data (AAAA-MM-DD) da primeira mensagem daquele atendimento.
- `hora_primeira_mensagem`: Hora da primeira mensagem (HH:MM:SS).
- `tempo_primeira_resposta`: Tempo entre a primeira mensagem do solicitante e a primeira resposta do Help Desk (em minutos e segundos).
- `tempo_total_atendimento`: Tempo entre a primeira e a última mensagem desse atendimento (também em minutos e segundos).
- `problema_relatado`: Resumo breve do problema relatado (com suas próprias palavras).

5. Se nenhuma solicitação de atendimento técnico for encontrada na conversa, **retorne uma lista vazia**:

```json
[]
"""

enc = tiktoken.encoding_for_model("gpt-4o-mini")
instruction_tokens = len(enc.encode(prompt_instructions))
print(f"Tokens do prompt fixo (instruções): {instruction_tokens}")

Tokens do prompt fixo (instruções): 458


In [11]:
def estimate_total_tokens(conversa_text, model="gpt-4o-mini"):
    enc = tiktoken.encoding_for_model(model)
    full_prompt = prompt_instructions + "\n\n" + conversa_text
    return len(enc.encode(full_prompt))


grouped_conversations['TotalTokens'] = grouped_conversations['ConversationHistory'].apply(estimate_total_tokens)
print(grouped_conversations['TotalTokens'].describe())

count      157.000000
mean      1580.006369
std       1545.758309
min        488.000000
25%        794.000000
50%       1187.000000
75%       1767.000000
max      14459.000000
Name: TotalTokens, dtype: float64


'[2025-03-11 14:39:24 UTC] Jessica Paloma Cunha Ribeiro Aversa <jessicaaversa@pecege.com>: Olá'

'Hellen Claudia Martins Meireles\x00 <hellenmeireles@pecege.com\x00>'

'28:25c0ac13-af11-43d3-ac26-9758831eb238\x00 <None>; Help Desk Pecege\x00 <helpdesk@pecege.com\x00>'

b'<html>\n<head>\n<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>\n</head>\n<body>\n<p>Maravilhaa!</p>\n<p>\xc2\xa0</p>\n<blockquote itemid="1741802902226" itemscope="" itemtype="http://schema.skype.com/Reply">\n<strong itemid="8:orgid:3531bcee-31c4-42bf-81cd-09c7af7d02a3" itemprop="mri">Help Desk Pecege</strong><span itemid="1741802902226" itemprop="time"></span>\n<p itemprop="preview">Boa tarde, Hellen, tudo bem e com voc\xc3\xaa? Barbi aqui</p>\n</blockquote>\n<p>\xc2\xa0</p>\n<p>Tudo certo por aqui tamb\xc3\xa9m, obrigadaa!</p>\n</body>\n</html>\n'